In [41]:
import pandas as pd
from data_analysis_utils import Univariate


In [42]:
dataset = pd.read_csv("PrePlacement.csv")
dataset.drop('sl_no', axis=1, inplace=True)
dataset


,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,0.0
4,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,M,80.60,Others,82.00,Others,Commerce,77.60,Comm&Mgmt,No,91.0,Mkt&Fin,74.49,Placed,400000.0
211,M,58.00,Others,60.00,Others,Science,72.00,Sci&Tech,No,74.0,Mkt&Fin,53.62,Placed,275000.0
212,M,67.00,Others,67.00,Others,Commerce,73.00,Comm&Mgmt,Yes,59.0,Mkt&Fin,69.72,Placed,295000.0
213,F,74.00,Others,66.00,Others,Commerce,58.00,Comm&Mgmt,No,70.0,Mkt&HR,60.23,Placed,204000.0


In [43]:
dataset.isnull().sum()


gender            0
ssc_p             0
ssc_b             0
hsc_p             0
hsc_b             0
hsc_s             0
degree_p          0
degree_t          0
workex            0
etest_p           0
specialisation    0
mba_p             0
status            0
salary            0
dtype: int64

### Covariance

In [44]:
quan, qual = Univariate.quanQual(dataset)

dataset[quan].cov()


,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
ssc_p,117.228377,58.853253,42.702550,37.659225,24.535952,9.088585e+05
hsc_p,58.853253,112.063731,33.684453,33.838355,21.517688,7.310079e+05
degree_p,42.702550,33.684453,53.604710,22.078774,17.185200,4.663363e+05
etest_p,37.659225,33.838355,22.078774,176.251018,16.886973,3.727004e+05
mba_p,24.535952,21.517688,17.185200,16.886973,34.028376,1.239934e+05
salary,908858.485818,731007.850848,466336.264888,372700.449468,123993.387361,2.259185e+10


In [45]:
# Covariance between degree_p and etest_p
dataset[['degree_p', 'etest_p']].cov()

# Var(degree_p) = 53.60 → degree_p values deviate from their mean with variance ~53.6.
# Var(etest_p) = 176.25 → etest_p has much larger variance — meaning its values are more spread out

# Covariance between degree_p and etest_p is positive (22.078774), so when degree_p increases, etest_p tends to increase too (and vice versa).


,degree_p,etest_p
degree_p,53.604710,22.078774
etest_p,22.078774,176.251018


In [46]:
# Covariance between mba_p and etest_p
dataset[['mba_p', 'etest_p']].cov()

# Var(mba_p) = 34.02 → mba_p values deviate from their mean with variance ~34.02.
# Var(etest_p) = 176.25 → etest_p has much larger variance — meaning its values are more spread out

# Covariance between mba_p and etest_p is positive (16.886973), so when mba_p increases, etest_p tends to increase too (and vice versa).


,mba_p,etest_p
mba_p,34.028376,16.886973
etest_p,16.886973,176.251018


### Correlation

In [47]:
dataset[quan].corr()


,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
ssc_p,1.000000,0.513478,0.538686,0.261993,0.388478,0.558475
hsc_p,0.513478,1.000000,0.434606,0.240775,0.348452,0.459424
degree_p,0.538686,0.434606,1.000000,0.227147,0.402376,0.423762
etest_p,0.261993,0.240775,0.227147,1.000000,0.218055,0.186775
mba_p,0.388478,0.348452,0.402376,0.218055,1.000000,0.141417
salary,0.558475,0.459424,0.423762,0.186775,0.141417,1.000000


In [48]:
# Correlation between mba_p and salary
dataset[['mba_p', 'salary']].corr()

# This means there is a weak positive correlation between mba_p and salary, means lots of variation in salary that’s unrelated to MBA score.


,mba_p,salary
mba_p,1.000000,0.141417
salary,0.141417,1.000000


### Multi Collinearity



In [49]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculating VIF
def calc_vif(X):

    vif = pd.DataFrame()
    
    # Create empty dataframe vif and column 'variables' to store the column names
    
    vif["variables"] = X.columns

    # Create column 'VIF' and calculate VIF for each feature
    # X.values converts dataframe to numpy array.
    # The for loop loops for each column in dataset, then that is used to calculate VIF and returned as list in vif dataframe
    
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    return(vif)

# Use only quantitative columns for VIF calculation
# Removing the output variable - using only input variables to check Multi Collinearity
quan.remove('salary')

calc_vif(dataset[quan])


# VIF > 10 → Potential problem; strong multicollinearity.
# VIF between 5 and 10 → Possible concern.
# VIF < 5 → Usually fine.

# The result shows that all input features have high VIF values, indicating potential multicollinearity issues.


,variables,VIF
0,ssc_p,67.087602
1,hsc_p,59.100713
2,degree_p,114.085694
3,etest_p,32.704858
4,mba_p,99.767641


Ways to remove multi collinearity:

1. Remove redundant variables
If two variables are telling almost the same story, drop one. Using the highest ones first

In [50]:
quan


['ssc_p', 'hsc_p', 'degree_p', 'etest_p', 'mba_p']

In [51]:
quan.remove('degree_p')
calc_vif(dataset[quan])


,variables,VIF
0,ssc_p,58.772895
1,hsc_p,56.420306
2,etest_p,32.091188
3,mba_p,70.874469


In [52]:
quan.remove('mba_p')
calc_vif(dataset[quan])


,variables,VIF
0,ssc_p,48.578843
1,hsc_p,47.552000
2,etest_p,27.084097


In [ ]:
quan.remove('ssc_p')
calc_vif(dataset[quan])

# The result shows I can use hsc_p or etest_p since both has same VIF values.


,variables,VIF
0,hsc_p,23.134453
1,etest_p,23.134453



2. Combine correlated variables
Merge them into a single variable (e.g., average of scores, or a weighted index).

Example: Create an “overall_academic_score” from ssc_p, hsc_p, and degree_p.



In [54]:
dataset['overall_academic_score'] = dataset[['ssc_p', 'hsc_p', 'degree_p']].mean(axis=1)


In [ ]:
data_vif = dataset[['overall_academic_score', 'mba_p', 'etest_p']]

calc_vif(data_vif)

# The result shows the multicollinearity has reduced when compared to initial analysis.


,variables,VIF
0,overall_academic_score,90.356407
1,mba_p,89.265958
2,etest_p,32.626720


3. Transform variables with PCA (Principal Component Analysis)
PCA turns correlated variables into a smaller set of uncorrelated components.

4. Use regularization techniques
Ridge Regression: Adds a penalty to large coefficients, which helps handle collinearity.

Lasso Regression: Can actually zero out redundant variables.

5. Collect more data
More observations can make correlations less problematic by giving the model more information.

6. Change the model
Use Partial Least Squares (PLS), Decision Trees, or other algorithms less sensitive to multicollinearity.